In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week6-lesson-1"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()


#### Sampling ratio for inferSchema - controls how much data is read

In [2]:
orders_df1 = spark.read \
.format('csv') \
.option("inferSchema","true") \
.option("samplingRatio",0.1) \
.load('/public/trendytech/datasets/orders_sample.csv')

In [3]:
orders_df1.show(3)

+---+--------------------+-----+---------------+
|_c0|                 _c1|  _c2|            _c3|
+---+--------------------+-----+---------------+
|  1|2013-07-25 00:00:...|11599|         CLOSED|
|  2|2013-07-25 00:00:...|  256|PENDING_PAYMENT|
|  3|2013-07-25 00:00:...|12111|       COMPLETE|
+---+--------------------+-----+---------------+
only showing top 3 rows



In [4]:
orders_df1.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: integer (nullable = true)
 |-- _c3: string (nullable = true)



## Enforce Schema

#### Schema DDL

In [5]:
order_schema= 'order_id long, order_date string , customer_id long, order_status string'

In [6]:
orders_df = spark.read \
.format('csv') \
.schema(order_schema) \
.load('/public/trendytech/datasets/orders_sample.csv')

In [7]:
orders_df.show(3)

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11599|         CLOSED|
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
+--------+--------------------+-----------+---------------+
only showing top 3 rows



In [8]:
orders_df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)



#### Struct Type

In [9]:
from pyspark.sql.types import *

In [10]:
structSchema = StructType ([
    StructField("order_id",LongType()),
    StructField("order_date",StringType()),
    StructField("customer_id",LongType()),
    StructField("order_status",StringType())
])

In [11]:
orders_df2 = spark.read \
.format('csv') \
.schema(structSchema) \
.load('/public/trendytech/datasets/orders_sample.csv')

In [12]:
orders_df2.show(3)

+--------+--------------------+-----------+---------------+
|order_id|          order_date|customer_id|   order_status|
+--------+--------------------+-----------+---------------+
|       1|2013-07-25 00:00:...|      11599|         CLOSED|
|       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|      12111|       COMPLETE|
+--------+--------------------+-----------+---------------+
only showing top 3 rows



In [13]:
orders_df2.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)



#### DateType

In [14]:
!hadoop fs -head /public/trendytech/datasets/orders_sample2.csv

1,07-25-2013,11599,CLOSED
2,07-25-2013,256,PENDING_PAYMENT
3,07-25-2013,12111,COMPLETE
4,07-25-2013,8827,CLOSED
5,07-25-2013,11318,COMPLETE
6,07-25-2013,7130,COMPLETE
7,07-25-2013,4530,COMPLETE
8,07-25-2013,2911,PROCESSING
9,07-25-2013,5657,PENDING_PAYMENT
10,07-25-2013,5648,PENDING_PAYMENT


In [23]:
## date is in mm-dd-yyyy format

In [26]:
structSchema1 = StructType ([
    StructField("order_id",LongType()),
    StructField("order_date",DateType()),
    StructField("customer_id",LongType()),
    StructField("order_status",StringType())
])

In [19]:
orders_df3 = spark.read \
.format('csv') \
.schema(structSchema1) \
.load('/public/trendytech/datasets/orders_sample2.csv')

In [ ]:
orders_df3.show(4) ## error loading due to date format- default is yyyy-mm-dd

In [21]:
orders_df3.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)



In [30]:
## fix for date -- add .option("dateFormat","mm-dd-yyyy") 

In [28]:
orders_df3 = spark.read \
.format('csv') \
.schema(structSchema1) \
.option("dateFormat","mm-dd-yyyy") \
.load('/public/trendytech/datasets/orders_sample2.csv')

In [29]:
orders_df3.show(4) 

+--------+----------+-----------+---------------+
|order_id|order_date|customer_id|   order_status|
+--------+----------+-----------+---------------+
|       1|2013-01-25|      11599|         CLOSED|
|       2|2013-01-25|        256|PENDING_PAYMENT|
|       3|2013-01-25|      12111|       COMPLETE|
|       4|2013-01-25|       8827|         CLOSED|
+--------+----------+-----------+---------------+
only showing top 4 rows



In [37]:
# another option - load as string. later convert using to date()

In [39]:
structSchema2 = StructType ([
    StructField("order_id",LongType()),
    StructField("order_date",StringType()),
    StructField("customer_id",LongType()),
    StructField("order_status",StringType())
])

In [40]:
orders_df4 = spark.read \
.format('csv') \
.schema(structSchema2) \
.load('/public/trendytech/datasets/orders_sample2.csv')

In [41]:
orders_df4.show(4) 

+--------+----------+-----------+---------------+
|order_id|order_date|customer_id|   order_status|
+--------+----------+-----------+---------------+
|       1|07-25-2013|      11599|         CLOSED|
|       2|07-25-2013|        256|PENDING_PAYMENT|
|       3|07-25-2013|      12111|       COMPLETE|
|       4|07-25-2013|       8827|         CLOSED|
+--------+----------+-----------+---------------+
only showing top 4 rows



In [42]:
orders_df4.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)



In [45]:
from pyspark.sql.functions import *

In [48]:
orders_df5 = orders_df4.withColumn('order_date1',to_date("order_date","mm-dd-yyyy"))

In [49]:
orders_df5.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date1: date (nullable = true)



In [50]:
orders_df5.show(4)

+--------+----------+-----------+---------------+-----------+
|order_id|order_date|customer_id|   order_status|order_date1|
+--------+----------+-----------+---------------+-----------+
|       1|07-25-2013|      11599|         CLOSED| 2013-01-25|
|       2|07-25-2013|        256|PENDING_PAYMENT| 2013-01-25|
|       3|07-25-2013|      12111|       COMPLETE| 2013-01-25|
|       4|07-25-2013|       8827|         CLOSED| 2013-01-25|
+--------+----------+-----------+---------------+-----------+
only showing top 4 rows



### nulls

In [51]:
!hadoop fs -head /public/trendytech/datasets/orders_sample3.csv

1,2013-07-25,11599,CLOSED
2,2013-07-25,256,PENDING_PAYMENT
3,2013-07-25,12111,COMPLETE
4,2013-07-25,8827,CLOSED
5,2013-07-25,11318,COMPLETE
6,2013-07-25,7130,COMPLETE
7,2013-07-25,error,COMPLETE
8,2013-07-25,2911,PROCESSING
9,2013-07-25,unknown,PENDING_PAYMENT
10,2013-07-25,5648,PENDING_PAYMENT


In [52]:
## invalid data types is some columns

In [53]:
orders_df5 = spark.read \
.format('csv') \
.schema(structSchema1) \
.load('/public/trendytech/datasets/orders_sample3.csv')

In [54]:
orders_df5.show()

+--------+----------+-----------+---------------+
|order_id|order_date|customer_id|   order_status|
+--------+----------+-----------+---------------+
|       1|2013-07-25|      11599|         CLOSED|
|       2|2013-07-25|        256|PENDING_PAYMENT|
|       3|2013-07-25|      12111|       COMPLETE|
|       4|2013-07-25|       8827|         CLOSED|
|       5|2013-07-25|      11318|       COMPLETE|
|       6|2013-07-25|       7130|       COMPLETE|
|       7|2013-07-25|       null|       COMPLETE|
|       8|2013-07-25|       2911|     PROCESSING|
|       9|2013-07-25|       null|PENDING_PAYMENT|
|      10|2013-07-25|       5648|PENDING_PAYMENT|
+--------+----------+-----------+---------------+



In [57]:
orders_df5.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)



In [58]:
## nulls loaded when invalid data type encountered in customer_id col

## Read modes

#### permissive - default option - as seen above. nulls loaded to col which has invalide data type. no error. other columns loaded

#### failfast

In [63]:
orders_df5 = spark.read \
.format('csv') \
.schema(structSchema1) \
.option("mode","failfast") \
.load('/public/trendytech/datasets/orders_sample3.csv')

In [ ]:
orders_df5.show(10)

In [67]:
#Py4JJavaError: An error occurred while calling o220.showString.
#: org.apache.spark.SparkException: Job aborted due to stage failure: 
# Task 0 in stage 16.0 failed 4 times, most recent failure: 
#     Lost task 0.3 in stage 16.0 (TID 25) (w01.itversity.com executor 2): 
#         org.apache.spark.SparkException: 
#             Malformed records are detected in record parsing. 
#             Parse Mode: FAILFAST. To process malformed records as null result, try setting the option 'mode' as 'PERMISSIVE'.

#### dropmalformed

In [70]:
orders_df6 = spark.read \
.format('csv') \
.schema(structSchema1) \
.option("mode","dropmalformed") \
.load('/public/trendytech/datasets/orders_sample3.csv')

In [71]:
orders_df6.show(10)

+--------+----------+-----------+---------------+
|order_id|order_date|customer_id|   order_status|
+--------+----------+-----------+---------------+
|       1|2013-07-25|      11599|         CLOSED|
|       2|2013-07-25|        256|PENDING_PAYMENT|
|       3|2013-07-25|      12111|       COMPLETE|
|       4|2013-07-25|       8827|         CLOSED|
|       5|2013-07-25|      11318|       COMPLETE|
|       6|2013-07-25|       7130|       COMPLETE|
|       8|2013-07-25|       2911|     PROCESSING|
|      10|2013-07-25|       5648|PENDING_PAYMENT|
+--------+----------+-----------+---------------+



In [73]:
## row 7 and 9 skipped as it contains a column with invalid data type